In [2]:
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from collections import Counter

In [5]:
# ── 1. Load data ──────────────────────────────────────────────
csv_path = 'synthetic_logs.csv'
df = pd.read_csv(csv_path)

print(f"Loaded {len(df)} log messages")
print(f"Columns: {list(df.columns)}")
print(f"Sources: {df['source'].unique()}")
print()

Loaded 2410 log messages
Columns: ['timestamp', 'source', 'log_message', 'target_label', 'complexity']
Sources: ['ModernCRM' 'AnalyticsEngine' 'ModernHR' 'BillingSystem' 'ThirdPartyAPI'
 'LegacyCRM']



In [6]:
# ── 2. Generate embeddings using SentenceTransformer ──────────
print("Loading SentenceTransformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

log_messages = df['log_message'].astype(str).tolist()
print(f"Generating embeddings for {len(log_messages)} log messages...")
embeddings = model.encode(log_messages, show_progress_bar=True, batch_size=32)
print(f"Embedding shape: {embeddings.shape}")
print()

Loading SentenceTransformer model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings for 2410 log messages...


Batches:   0%|          | 0/76 [00:00<?, ?it/s]

Embedding shape: (2410, 384)



In [12]:
# ── 3. Cluster with DBSCAN ───────────────────────────────────
print("Running DBSCAN clustering...")
dbscan = DBSCAN(eps=0.2, min_samples=1, metric='cosine')
cluster_labels = dbscan.fit_predict(embeddings)

df['cluster'] = cluster_labels
df.head()

Running DBSCAN clustering...


,timestamp,source,log_message,target_label,complexity,cluster
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0


In [13]:
df[df.cluster==1]

,timestamp,source,log_message,target_label,complexity,cluster
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
10,8/9/2025 18:58,ModernCRM,Email server encountered a sending fault,Error,bert,1
217,1/22/2025 5:45,BillingSystem,Mail service encountered a delivery glitch,Error,bert,1
248,5/2/2025 23:04,ModernHR,Service disruption caused by email sending error,Critical Error,bert,1
265,3/30/2025 23:53,ModernCRM,Email system had a problem sending emails,Error,bert,1
361,11/19/2025 23:06,BillingSystem,Email service experienced a sending issue,Error,bert,1
450,10/27/2025 5:59,ThirdPartyAPI,Email delivery system encountered an error,Error,bert,1
477,12/2/2025 10:30,AnalyticsEngine,Email transmission error caused service impact,Critical Error,bert,1
570,11/7/2025 18:08,ThirdPartyAPI,Email service impacted by sending failure,Critical Error,bert,1
678,4/28/2025 15:13,AnalyticsEngine,Email delivery problem affected system,Critical Error,bert,1


In [16]:
pd.set_option('display.max_colwidth', None)
cluster_counts = df['cluster'].value_counts()
large_clusters = cluster_counts[cluster_counts > 10].index
for cluster in large_clusters:
  print(f"Cluster {cluster}:")
  print(df[df['cluster'] == cluster]['log_message'].head(5).to_string(index=False))
  print()

Cluster 0:
           nova.osapi_compute.wsgi.server [req-b9718cd8-f65e-49cc-8349-6cf7122af137 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" status: 200 len: 1893 time: 0.2675118
nova.osapi_compute.wsgi.server [req-4895c258-b2f8-488f-a2a3-4fae63982e48 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" HTTP status code -  200 len: 211 time: 0.0968180
            nova.osapi_compute.wsgi.server [req-ee8bc8ba-9265-4280-9215-dbe000a41209 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" RCODE  200 len: 1874 time: 0.2280791
      nova.osapi_compute.wsgi.server [req-f0bffbc3-5ab0-4916-91c1-0a61dd7d4ec2 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v

In [26]:
import re

def classify_with_regex(log_messages):
  regex_patterns = {
      r"nova\.osapi_compute\.wsgi\.server.*GET \/v2\/[a-f0-9\-]+\/servers\/detail HTTP\/1\.1.*(status|HTTP status code|RCODE|Return code):?\s*200.*time:\s*[0-9\.]+": "HTTP Status",
      r"nova\.compute\.claims.*(Total memory|memory limit|disk limit|vcpu limit).*": "Resource Usage",
      r"User\s+\w+\s+logged\s+(in|out)\.": "User Action",
      r"Backup (started|ended) at .*": "System Notification",
      r".*(Multiple|multiple).*(login|incorrect).*(attempts|failures).*": "Security Alert",
      r"Backup completed successfully\.": "System Notification",
      r"System updated to version \d+\.\d+\.\d+\.": "System Notification",
      r".*(replication|Replication).*(failed|failure|did not complete).*": "Error",
      r"File .* uploaded successfully by user .*": "System Notification",
      r".*(Denied access|Unauthorized login|blocked|Invalid login attempt).*": "Security Alert",
      r".*(Critical system|malfunction|Failure occurred).*component.*ID .*": "Critical Error",
      r"Disk cleanup completed successfully\.": "System Notification",
      r"System reboot initiated by user .*": "System Notification",
      r"User \d+.*(unauthorized|failed to provide valid API|bypass API|without proper authorization).*": "Security Alert",
      r"Account with ID .* created by .*": "User Action",
      r".*(Email|Mail).*(issue|error|fault|glitch|problem).*": "Error",
      r"nova\.compute\.resource_tracker.*(Final resource view|Total usable vcpus).*": "Resource Usage",
      r".*(security threat|suspicious activity|potential security incident|Anomalous activity).*": "Security Alert",
      r".*(invalid data format|format mismatch|input format|format validation).*": "Data Error",
      r".*(RAID|disk).*(failure|fault|crash|faulty).*": "Hardware Failure",
      r".*SSL certificate.*(failed|invalid|expired|validation).*": "Security Alert",
      r".*(kernel panic|kernel failure|boot process|boot sequence).*(failed|terminated|stopped).*": "Critical Error",
      r"User \d+.*escalated.*admin.*": "Security Alert",
      r".*configuration.*(invalid|corrupted|failure|error).*": "Error",
      r".*(admin privilege escalation|Potential security threat).*": "Security Alert",
      r".*(privilege elevation|elevated admin privileges).*": "Security Alert"
  }
  for pattern, label in regex_patterns.items():
    if re.search(pattern, log_messages, re.IGNORECASE):
      return label
  return None


In [27]:
classify_with_regex("User user1234 logged OUT.")

'User Action'

In [28]:
df['regex_label'] = df['log_message'].apply(classify_with_regex)

In [29]:
df

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
0,2025-06-27 07:20:25,ModernCRM,"nova.osapi_compute.wsgi.server [req-b9718cd8-f65e-49cc-8349-6cf7122af137 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 ""GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1"" status: 200 len: 1893 time: 0.2675118",HTTP Status,bert,0,HTTP Status
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1,Error
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2,None
3,2025-07-12 00:24:16,ModernHR,"nova.osapi_compute.wsgi.server [req-4895c258-b2f8-488f-a2a3-4fae63982e48 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 ""GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1"" HTTP status code - 200 len: 211 time: 0.0968180",HTTP Status,bert,0,None
4,2025-06-02 18:25:23,BillingSystem,"nova.osapi_compute.wsgi.server [req-ee8bc8ba-9265-4280-9215-dbe000a41209 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 ""GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1"" RCODE 200 len: 1874 time: 0.2280791",HTTP Status,bert,0,HTTP Status
...,...,...,...,...,...,...,...
2405,2025-08-13 07:29:25,ModernHR,"nova.osapi_compute.wsgi.server [req-96c3ec98-21a0-4af2-84a8-d4989512413e 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 ""GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1"" Return code: 200 len: 1916 time: 0.2677610",HTTP Status,bert,0,HTTP Status
2406,1/11/2025 5:32,ModernHR,User 3844 account experienced multiple failed logins,Security Alert,bert,7,None
2407,2025-08-03 03:07:47,ThirdPartyAPI,"nova.metadata.wsgi.server [req-b6d4a270-accb-4c3a-8179-9611e52e1768 - - - - -] 10.11.21.124,10.11.10.1 ""GET /openstack/2013-10-17 HTTP/1.1"" RCODE 200 len: 157 time: 0.2249990",HTTP Status,bert,0,None
2408,11/11/2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,bert,1,None


In [32]:
df[df.regex_label.notnull()]

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
0,2025-06-27 07:20:25,ModernCRM,"nova.osapi_compute.wsgi.server [req-b9718cd8-f65e-49cc-8349-6cf7122af137 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 ""GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1"" status: 200 len: 1893 time: 0.2675118",HTTP Status,bert,0,HTTP Status
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1,Error
4,2025-06-02 18:25:23,BillingSystem,"nova.osapi_compute.wsgi.server [req-ee8bc8ba-9265-4280-9215-dbe000a41209 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 ""GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1"" RCODE 200 len: 1874 time: 0.2280791",HTTP Status,bert,0,HTTP Status
5,2025-10-09 10:30:31,ModernHR,"nova.osapi_compute.wsgi.server [req-f0bffbc3-5ab0-4916-91c1-0a61dd7d4ec2 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 ""GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1"" Return code: 200 len: 1874 time: 0.2131531",HTTP Status,bert,0,HTTP Status
6,3/1/2025 19:14,ModernHR,Shard 6 replication task ended in failure,Error,bert,3,Error
...,...,...,...,...,...,...,...
2400,2025-01-07 09:13:28,ThirdPartyAPI,"nova.compute.resource_tracker [req-addc1839-2ed5-4778-b57e-5854eb7b8b09 - - - - -] Total usable vcpus: 16, total allocated vcpus: 1",Resource Usage,bert,10,Resource Usage
2401,2025-12-05 15:51:51,ModernCRM,"nova.osapi_compute.wsgi.server [req-4bdf00b0-3bbc-44fa-bd84-de85ec43c8a2 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 ""GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1"" status: 200 len: 1916 time: 0.2769570",HTTP Status,bert,0,HTTP Status
2403,10/1/2025 1:31,ModernCRM,Backup completed successfully.,System Notification,regex,8,System Notification
2404,2025-09-18 02:18:30,ThirdPartyAPI,"nova.osapi_compute.wsgi.server [req-2c9c783f-3c7a-4844-87b1-e207a38c74ab 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 ""GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1"" Return code: 200 len: 1893 time: 0.2527661",HTTP Status,bert,0,HTTP Status


In [33]:
df_non_regex = df[df.regex_label.isnull()].copy()
df_non_regex.shape

(1141, 7)

In [34]:
print(df_non_regex['target_label'].value_counts()[df_non_regex['target_label'].value_counts() <=5].index.tolist())

['Workflow Error', 'Deprecation Warning']


In [38]:
df_non_legacy = df_non_regex[df_non_regex['source']!='LegacyCRM']
df_non_legacy.source.unique()

array(['AnalyticsEngine', 'ModernHR', 'BillingSystem', 'ThirdPartyAPI',
       'ModernCRM'], dtype=object)

In [40]:
log_messages = df_non_legacy['log_message'].astype(str).tolist()
print(f"Generating embeddings for df_non_legacy {len(log_messages)} log messages...")
filtered_embeddings = model.encode(log_messages, show_progress_bar=True, batch_size=32)
print(f"Embedding shape: {filtered_embeddings.shape}")
print()

Generating embeddings for df_non_legacy 1134 log messages...


Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Embedding shape: (1134, 384)



In [42]:
X = filtered_embeddings
y = df_non_legacy['target_label']

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
report = classification_report(y_test, y_pred)
print(report)

                precision    recall  f1-score   support

Critical Error       1.00      0.95      0.98        42
         Error       0.95      1.00      0.97        39
   HTTP Status       1.00      1.00      1.00       180
Resource Usage       1.00      1.00      1.00        20
Security Alert       1.00      1.00      1.00        60

      accuracy                           0.99       341
     macro avg       0.99      0.99      0.99       341
  weighted avg       0.99      0.99      0.99       341



In [44]:
import joblib
import os

# Create the directory if it doesn't exist
if not os.path.exists('models'):
    os.makedirs('models')

joblib.dump(clf, 'models/jobclassifier.joblib')

['models/jobclassifier.joblib']

In [71]:
# First, let's create a dummy .py file to test
with open('bert_processor.py', 'r') as f:
    print('bert_processor.py read successfully.')
    !python bert_processor.py

bert_processor.py created successfully.
Loading weights: 100% 103/103 [00:00<00:00, 1176.21it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
alpha.osapi_compute.wsgi.server - 12.10.11.1 - API returned 404 not found error -> HTTP Status
GET /v2/3454/servers/detail HTTP/1.1 RCODE 404 len: 1583 time: 0.1878400 -> HTTP Status
System crashed due to drivers errors when restarting the server -> Critical Error
Hey bro, chill ya! -> Unclassified
Multiple login failures occurred on user 6454 account -> Security Alert
Server A790 was restarted unexpectedly during the process of data transfer -> Error


In [90]:
with open('llm_processor.py', 'r') as f:
    print('llm_processor.py read successfully.')
    !python llm_processor.py

llm_processor.py read successfully.
Workflow Error
Deprecation Warning
Unclassified


In [101]:
import llm_processor
import bert_processor

def classify(logs):
  Labels = []
  for source, log_msg in logs:
    Label = classify_log(source, log_msg)
    Labels.append(Label)
  return Labels

def classify_log(source, log_message):
  if source == "LegacyCRM":
    # Using the imported module instead of opening the file as text
    return llm_processor.classify_with_llm(log_message)
  else:
    label = classify_with_regex(log_message)
    if label is None:
      return bert_processor.classify_with_bert(log_message)

    return label


In [111]:
def classify_csv(input_file):
    import pandas as pd
    df = pd.read_csv(input_file)

    # Perform classification
    df["target_label"] = classify(list(zip(df["source"], df["log_message"])))

    # Save the modified file
    output_file = "resources/output.csv"
    df.to_csv(output_file, index=False)

    return output_file

In [112]:
if __name__ == '__main__':
    # This function already reads the CSV and runs classification
    output_path = classify_csv("test.csv")

    print(f"Classification complete. Results saved to: {output_path}")

    # Test with the manual list
    logs = [
        ("ModernCRM", "IP 192.168.133.114 blocked due to potential attack"),
        ("BillingSystem", "User 12345 logged in."),
        ("AnalyticsEngine", "File data_6957.csv uploaded successfully by user User265."),
        ("AnalyticsEngine", "Backup completed successfully."),
        ("ModernHR", "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1 RCODE  200 len: 1583 time: 0.1878400"),
        ("ModernHR", "Admin access escalation detected for user 9429"),
        ("LegacyCRM", "Case escalation for ticket ID 7324 failed because the assigned support agent is no longer active."),
        ("LegacyCRM", "Invoice generation process aborted for order ID 8910 due to invalid tax calculation module."),
        ("LegacyCRM", "The 'BulkEmailSender' feature is no longer supported. Use 'EmailCampaignManager' for improved functionality."),
        ("LegacyCRM", " The 'ReportGenerator' module will be retired in version 4.0. Please migrate to the 'AdvancedAnalyticsSuite' by Dec 2025")
    ]

    labels = classify(logs)

    print("\n--- Manual Logs Classification Results ---")
    for log, label in zip(logs, labels):
        print(f"{log[0]} -> {label}")

    # To see the CSV results, we should read the output file
    print("\n--- CSV Logs Classification Results (First 5) ---")
    import pandas as pd
    print(pd.read_csv(output_path)[['source', 'target_label']])

Classification complete. Results saved to: resources/output.csv

--- Manual Logs Classification Results ---
ModernCRM -> Security Alert
BillingSystem -> User Action
AnalyticsEngine -> System Notification
AnalyticsEngine -> System Notification
ModernHR -> HTTP Status
ModernHR -> Security Alert
LegacyCRM -> Workflow Error
LegacyCRM -> Workflow Error
LegacyCRM -> Deprecation Warning
LegacyCRM -> Deprecation Warning

--- CSV Logs Classification Results (First 5) ---
            source         target_label
0        ModernCRM       Security Alert
1    BillingSystem          User Action
2  AnalyticsEngine  System Notification
3  AnalyticsEngine  System Notification
4         ModernHR          HTTP Status
5         ModernHR       Security Alert
6        LegacyCRM       Workflow Error
7        LegacyCRM       Workflow Error
8        LegacyCRM  Deprecation Warning
9        LegacyCRM  Deprecation Warning


In [73]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.5 MB/s eta 0:00:00
